**BERT**

In [ ]:
!pip install Korpora

In [ ]:
#네이버 영화 리뷰 데이터 불러오기
import numpy as np
import pandas as pd
from Korpora import Korpora

corpus=Korpora.load("nsmc")
df=pd.DataFrame(corpus.test).sample(20000, random_state=42)
train, valid, test=np.split(
    df.sample(frac=1, random_state=42), [int(0.6*len(df)), int(0.8*len(df))]
)

print(train.head(5).to_markdown())
print(f"Training Data Size: {len(train)}")
print(f"Validation Data Size: {len(valid)}")
print(f"Testing Data Size: {len(test)}")


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
#BERT 입력 텐서 생성
import torch
from transformers import BertTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler


def make_dataset(data, tokenizer, device):
    tokenized=tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )
    input_ids=tokenized["input_ids"].to(device)
    attention_mask=tokenized["attention_mask"].to(device)
    labels=torch.tensor(data.label.values, dtype=torch.long).to(device)
    return TensorDataset(input_ids, attention_mask, labels)


def get_dataloader(dataset, sampler, batch_size):
    data_sampler=sampler(dataset)
    dataloader=DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

epochs=5
batch_size=32
device="cuda" if torch.cuda.is_available() else "cpu"
tokenizer=BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    do_lower_case=False
)

train_dataset=make_dataset(train, tokenizer, device)
train_dataloader=get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset=make_dataset(valid, tokenizer, device)
valid_dataloader=get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset=make_dataset(test, tokenizer, device)
test_dataloader=get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

(tensor([   101,  58466,   9812, 118956, 119122,  59095,  10892,   9434, 118888,
           117,   9992,  40032,  30005,    117,   9612,  37824,   9410,  12030,
         42337,  10739,  83491,  12508,    106,    106,    102,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,

In [ ]:
#BERT 모델 선언
from torch import optim
from transformers import BertForSequenceClassification

model=BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
optimizer=optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("L", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("| L", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("| | L", sssub_name)

bert
L embeddings
| L word_embeddings
| L position_embeddings
| L token_type_embeddings
| L LayerNorm
| L dropout
L encoder
| L layer
| | L 0
| | L 1
| | L 2
| | L 3
| | L 4
| | L 5
| | L 6
| | L 7
| | L 8
| | L 9
| | L 10
| | L 11
L pooler
| L dense
| L activation
dropout
classifier


In [ ]:
import numpy as np
from torch import nn


def calc_accuracy(preds, labels):
    pred_flat=np.argmax(preds, axis=1).flatten()
    labels_flat=labels.flatten()
    return np.sum(pred_flat==labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss=0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs=model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        loss=outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss=train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion=nn.CrossEntropyLoss()
        val_loss, val_accuracy=0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs=model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            logits=outputs.logits

            loss=criterion(logits, labels)
            logits=logits.detach().cpu().numpy()
            label_ids=labels.to("cpu").numpy()
            accuracy=calc_accuracy(logits, label_ids)

            val_loss += loss
            val_accuracy += accuracy

    val_loss=val_loss/len(dataloader)
    val_accuracy=val_accuracy/len(dataloader)
    return val_loss, val_accuracy


best_loss=10000
for epoch in range(epochs):
    train_loss=train(model, optimizer, train_dataloader)
    val_loss, val_accuracy=evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss=val_loss
        torch.save(model.state_dict(), "../BertForSequenceClassification.pt")
        print("Saved the model weights")

Epoch 1: Train Loss: 0.5465 Val Loss: 0.4536 Val Accuracy 0.7875
Saved the model weights
Epoch 2: Train Loss: 0.4057 Val Loss: 0.4726 Val Accuracy 0.7893
Epoch 3: Train Loss: 0.3281 Val Loss: 0.4175 Val Accuracy 0.8173
Saved the model weights
Epoch 4: Train Loss: 0.2489 Val Loss: 0.4475 Val Accuracy 0.8185
Epoch 5: Train Loss: 0.1948 Val Loss: 0.5021 Val Accuracy 0.8185


In [ ]:
model=BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
model.load_state_dict(torch.load("../BertForSequenceClassification.pt"))

test_loss, test_accuracy=evaluation(model, test_dataloader)
print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss : 0.4269
Test Accuracy : 0.8113


**BART**

In [ ]:
!pip install datasets

In [1]:
!pip install --upgrade datasets huggingface_hub

In [ ]:
import numpy as np
from datasets import load_dataset

news=load_dataset("argilla/news-summary", split="test")
df=news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]
df["prediction"]=df["prediction"].map(lambda x: x[0]["text"])
train, valid, test=np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(f"Source News: {train.text.iloc[0][:200]}")
print(f"Summarization: {train.prediction.iloc[0][:50]}")
print(f"Training Data Size: {len(train)}")
print(f"Validation Data Size: {len(valid)}")
print(f"Testing Data Size: {len(test)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Source News: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, well-educated
Summarization: Putin says had useful interaction with Trump at Vi
Training Data Size: 3000
Validation Data Size: 1000
Testing Data Size: 1000


/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
#BART 입력 텐서 생성
import torch
from transformers import BartTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    tokenized=tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )
    labels=[]
    input_ids=tokenized["input_ids"].to(device)
    attention_mask=tokenized["attention_mask"].to(device)
    for target in data.prediction:
        labels.append(tokenizer.encode(target, return_tensors="pt").squeeze())
    labels=pad_sequence(labels, batch_first=True, padding_value=-100).to(device)
    return TensorDataset(input_ids, attention_mask, labels)

def get_datalodader(dataset, sampler, batch_size):
    data_sampler=sampler(dataset)
    dataloader=DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

epochs=5
batch_size=8
device="cuda" if torch.cuda.is_available() else "cpu"
tokenizer=BartTokenizer.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
)

train_dataset=make_dataset(train, tokenizer, device)
train_dataloader=get_datalodader(train_dataset, RandomSampler, batch_size)

valid_dataset=make_dataset(valid, tokenizer, device)
valid_dataloader=get_datalodader(valid_dataset, SequentialSampler, batch_size)

test_dataset=make_dataset(test, tokenizer, device)
test_dataloader=get_datalodader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


(tensor([   0,  495, 1889,  ...,    1,    1,    1], device='cuda:0'), tensor([1, 1, 1,  ..., 0, 0, 0], device='cuda:0'), tensor([    0, 35891,   161,    56,  5616, 10405,    19,   140,    23,  5490,
         3564,     2,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
       device='cuda:0'))


In [ ]:
#BART 모델 선언
from torch import optim
from transformers import BartForConditionalGeneration

model=BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
optimizer=optim.AdamW(model.parameters(), lr=5e-5, eps=1e-8)

In [ ]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("L", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("| L", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("| | L", sssub_name)

model
L shared
L encoder
| L embed_tokens
| L embed_positions
| L layers
| | L 0
| | L 1
| | L 2
| | L 3
| | L 4
| | L 5
| L layernorm_embedding
L decoder
| L embed_tokens
| L embed_positions
| L layers
| | L 0
| | L 1
| | L 2
| | L 3
| | L 4
| | L 5
| L layernorm_embedding
lm_head


In [ ]:
!pip install evaluate rouge_score absl-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=e70b5180f78388ff331d4494a467db32f4c31dff6b418c053f06449acbd41729
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
#BART 모델 학습 및 평가
'''
import numpy as np
import evaluate

def calc_rouge(preds, labels):
    preds=preds.argmax(axis=-1)
    labels=np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds=tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels=tokenizer.batch_decode(labels, skip_special_tokens=True)

    rouge2=rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return rouge2["rouge2"]

def train(model, optimizer, dataloader):
    model.train()
    train_loss=0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs=model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        loss=outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss=train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss, val_rouge=0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs=model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            logits=outputs.logits
            loss=outputs.loss

            logits=logits.detach().cpu().numpy()
            label_ids=labels.to("cpu").numpy()
            rouge=calc_rouge(logits, label_ids)

            val_loss += loss
            val_rouge += rouge

    val_loss=val_loss / len(dataloader)
    val_rouge=val_rouge / len(dataloader)
    return val_loss, val_rouge

rouge_score=evaluate.load("rouge", tokenizer=tokenizer)
best_loss=10000
for epoch in range(epochs):
    train_loss=train(model, optimizer, train_dataloader)
    val_loss, val_accuracy=evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Rouge {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "../BartForConditionalGeneration.pt")
        print("Saved the model weights")
'''

'\nimport numpy as np\nimport evaluate\n\ndef calc_rouge(preds, labels):\n    preds=preds.argmax(axis=-1)\n    labels=np.where(labels != -100, labels, tokenizer.pad_token_id)\n\n    decoded_preds=tokenizer.batch_decode(preds, skip_special_tokens=True)\n    decoded_labels=tokenizer.batch_decode(labels, skip_special_tokens=True)\n\n    rouge2=rouge_score.compute(\n        predictions=decoded_preds,\n        references=decoded_labels\n    )\n    return rouge2["rouge2"]\n\ndef train(model, optimizer, dataloader):\n    model.train()\n    train_loss=0.0\n\n    for input_ids, attention_mask, labels in dataloader:\n        outputs=model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)\n\n        loss=outputs.loss\n        train_loss += loss.item()\n\n        optimizer.zero_grad()\n        loss.backward()\n        optimizer.step()\n\n    train_loss=train_loss / len(dataloader)\n    return train_loss\n\ndef evaluation(model, dataloader):\n    with torch.no_grad():\n        mode

In [ ]:
#BART 모델 평가
'''
model=BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
model.load_state_dict(torch.load("../BartForConditionalGeneration.pt"))

test_loss, test_rouge_score=evaluation(model, test_dataloader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test ROUGE-2 Score: {test_rouge_score:.4f}")
'''

'\nmodel=BartForConditionalGeneration.from_pretrained(\n    pretrained_model_name_or_path="facebook/bart-base"\n).to(device)\nmodel.load_state_dict(torch.load("../BartForConditionalGeneration.pt"))\n\ntest_loss, test_rouge_score=evaluation(model, test_dataloader)\nprint(f"Test Loss: {test_loss:.4f}")\nprint(f"Test ROUGE-2 Score: {test_rouge_score:.4f}")\n'

In [ ]:
#문장 요약문 비교
from transformers import pipeline

summarizer=pipeline(
    task="summarization",
    model=model,
    tokenizer=tokenizer,
    max_length=54,
    device="cpu"
)

for index in range(5):
    news_text=test.text.iloc[index]
    summarization=test.prediction.iloc[index]
    predicted_summarization=summarizer(news_text)[0]["summary_text"]
    print(f"정답 요약문: {summarization}")
    print(f"모델 요약문: {predicted_summarization}\n")

Device set to use cpu
Your max_length is set to 128, but your input_length is only 111. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=55)


정답 요약문: Clinton leads Trump by 4 points in Washington Post: ABC News poll
모델 요약문: WASHINGTON (Reuters) - Democratic presidential nominee Hillary Clinton leads Republican Donald Trump by 4 percentage points in a four-way race for the Nov. 8 election, according to a Washington Post-ABC News opinion poll of likely voters released on Friday. Clinton had 47 percent support compared with Trump’s 43 percent in the poll conducted from Monday to Thursday, the Post said. It said Clinton’S lead was up from 3 points in the previous day“in the range of sampling error.”  

정답 요약문: Democrats question independence of Trump Supreme Court nominee
모델 요약문: WASHINGTON (Reuters) - Democratic U.S. senators on Monday sharpened a potential line of attack against Neil Gorsuch’s nomination to the Supreme Court by questioning whether he would be sufficiently independent as a justice in light of President Donald Trump‘s vigorous use of unilateral presidential power including his travel ban. Their comments came aft

Your max_length is set to 128, but your input_length is only 108. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=54)


정답 요약문: Romanian ruling party leader investigated over 'criminal group'
모델 요약문: BUCHAREST (Reuters) - Romanian anti-corruption prosecutors opened an investigation on  Monday into the leader of the ruling Social Democrat Party, Liviu Dragnea, on suspicion of forming a  criminal group  to siphon off cash from state projects, some of them EU-funded.    Romania’s parliament speaker has dismissed past investigations as politically motivated.  The prosecutors said Dragnea was suspected of forming an organized criminal group in 2001. They said there were suspicions the group was still active. The investigation focused on road construction firm Tel Drum SA, formerly controlled by the county council of the southern Teleorman

정답 요약문: Billionaire environmental activist Tom Steyer endorses Clinton
모델 요약문: WASHINGTON (Reuters) - Environmental activist Tom Steyer endorsed Hillary Clinton on Wednesday for U.S. president a day after she secured the Democratic nomination to run in the Nov. 8 election.

**ELECTRA**

In [ ]:
#네이버 영화 리뷰 데이터세트 전처리
import torch
from transformers import ElectraTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
    tokenized=tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )
    input_ids=tokenized["input_ids"].to(device)
    attention_mask=tokenized["attention_mask"].to(device)
    labels=torch.tensor(data.label.values, dtype=torch.long).to(device)
    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler=sampler(dataset)
    dataloader=DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

epochs=5
batch_size=32
device="cuda" if torch.cuda.is_available() else "cpu"
tokenizer=ElectraTokenizer.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    do_lower_case=False,
)

train_dataset=make_dataset(train, tokenizer, device)
train_dataloader=get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset=make_dataset(valid, tokenizer, device)
valid_dataloader=get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset=make_dataset(test, tokenizer, device)
test_dataloader=get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

(tensor([    2,  6511, 14347,  4087,  4665,  4112,  2924,  4806,    16,  3809,
         4309,  4275,    16,  3201,  4376,  2891,  4139,  4212,  4007,  6557,
         4200,     5,     5,     3,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0], device='cuda:0'), tensor([1, 1, 1, 1, 1, 1, 1, 

In [ ]:
#KoELECTRA 모델 선언
from torch import optim
from transformers import ElectraForSequenceClassification

model=ElectraForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)
optimizer=optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("L", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("| L", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("| | L", sssub_name)

electra
L embeddings
| L word_embeddings
| L position_embeddings
| L token_type_embeddings
| L LayerNorm
| L dropout
L encoder
| L layer
| | L 0
| | L 1
| | L 2
| | L 3
| | L 4
| | L 5
| | L 6
| | L 7
| | L 8
| | L 9
| | L 10
| | L 11
classifier
L dense
L activation
L dropout
L out_proj


In [ ]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat=np.argmax(preds, axis=1).flatten()
    labels_flat=labels.flatten()
    return np.sum(pred_flat==labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss=0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs=model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss=outputs.loss
        train_loss+=loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss=train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion=nn.CrossEntropyLoss()
        val_loss, val_accuracy=0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs=model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits=outputs.logits

            loss=criterion(logits, labels)
            logits=logits.detach().cpu().numpy()
            label_ids=labels.to("cpu").numpy()
            accuracy=calc_accuracy(logits, label_ids)

            val_loss+=loss
            val_accuracy+=accuracy

    val_loss=val_loss/len(dataloader)
    val_accuracy=val_accuracy/len(dataloader)
    return val_loss, val_accuracy


best_loss=10000
for epoch in range(epochs):
    train_loss=train(model, optimizer, train_dataloader)
    val_loss, val_accuracy=evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss=val_loss
        torch.save(model.state_dict(), "../ElectraForSequenceClassification.pt")
        print("Saved the model weights")

Epoch 1: Train Loss: 0.4471 Val Loss: 0.3152 Val Accuracy 0.8750
Saved the model weights
Epoch 2: Train Loss: 0.2829 Val Loss: 0.3032 Val Accuracy 0.8805
Saved the model weights
Epoch 3: Train Loss: 0.2135 Val Loss: 0.3215 Val Accuracy 0.8838
Epoch 4: Train Loss: 0.1601 Val Loss: 0.3331 Val Accuracy 0.8798
Epoch 5: Train Loss: 0.1168 Val Loss: 0.4372 Val Accuracy 0.8710


In [ ]:
model=ElectraForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)
model.load_state_dict(torch.load("../ElectraForSequenceClassification.pt"))

test_loss, test_accuracy=evaluation(model, test_dataloader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss: 0.3243
Test Accuracy: 0.8762


**T5**

In [2]:
#뉴스 요약 데이터세트 불러오기
import numpy as np
from datasets import load_dataset

news=load_dataset("argilla/news-summary", split="test")
df=news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]
df["text"]="summarize: " + df["text"]
df["prediction"]=df["prediction"].map(lambda x: x[0]["text"])
train, valid, test=np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(f"Source News: {train.text.iloc[0][:200]}")
print(f"Summarization: {train.prediction.iloc[0][:50]}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

(…)-00000-of-00001-ebc48879f34571f6.parquet:   0%|          | 0.00/1.54M [00:00<?, ?B/s]

(…)-00000-of-00001-6227bd8eb10a9b50.parquet:   0%|          | 0.00/31.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20417 [00:00<?, ? examples/s]

Source News: summarize: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, we
Summarization: Putin says had useful interaction with Trump at Vi


/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [4]:
#뉴스 요약 데이터세트 전처리
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    source=tokenizer(
        text=data.text.tolist(),
        padding="max_length",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    )
    target=tokenizer(
        text=data.prediction.tolist(),
        padding="max_length",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    )

    source_ids=source["input_ids"].squeeze().to(device)
    source_mask=source["attention_mask"].squeeze().to(device)
    target_ids=target["input_ids"].squeeze().to(device)
    target_mask=target["attention_mask"].squeeze().to(device)
    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

epochs=3
batch_size=8
device="cuda" if torch.cuda.is_available() else "cpu"
tokenizer=T5Tokenizer.from_pretrained(
    pretrained_model_name_or_path="t5-small"
)

train_dataset=make_dataset(train, tokenizer, device)
train_dataloader=get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset=make_dataset(valid, tokenizer, device)
valid_dataloader=get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset=make_dataset(test, tokenizer, device)
test_dataloader=get_dataloader(test_dataset, SequentialSampler, batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

[tensor([[21603,    10,  4800,  ...,     0,     0,     0],
        [21603,    10,   262,  ...,     7,     3,     1],
        [21603,    10,  8161,  ...,   105,  7238,     1],
        ...,
        [21603,    10,   549,  ...,     0,     0,     0],
        [21603,    10,    41,  ...,     0,     0,     0],
        [21603,    10,     3,  ..., 16923,     9,     1]], device='cuda:0'), tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), tensor([[  419, 17445,  1281,  ...,     0,     0,     0],
        [ 7449,  1299,     7,  ...,     0,     0,     0],
        [ 2523,    10,     3,  ...,     0,     0,     0],
        ...,
        [18263, 17579,  5752,  ...,     0,     0,     0],
        [ 2523,  4014,  2540,  ...,     0,     0,     0],
        [24463,    10,  3434,  ...,     0,     0,     0]], device='cuda:0'), ten

In [5]:
#T5 모델 선언
from torch import optim
from transformers import T5ForConditionalGeneration

model=T5ForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="t5-small",
).to(device)
optimizer=optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("L", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("| L", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("| | L", sssub_name)

shared
encoder
L embed_tokens
L block
| L 0
| | L layer
| L 1
| | L layer
| L 2
| | L layer
| L 3
| | L layer
| L 4
| | L layer
| L 5
| | L layer
L final_layer_norm
L dropout
decoder
L embed_tokens
L block
| L 0
| | L layer
| L 1
| | L layer
| L 2
| | L layer
| L 3
| | L layer
| L 4
| | L layer
| L 5
| | L layer
L final_layer_norm
L dropout
lm_head


In [7]:
#T5 모델 학습 및 평가
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat=np.argmax(preds, axis=1).flatten()
    labels_flat=labels.flatten()
    return np.sum(pred_flat==labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss=0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids=target_ids[:, :-1].contiguous()
        labels=target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:]==tokenizer.pad_token_id]=-100

        outputs=model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
        )

        loss=outputs.loss
        train_loss+=loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss=train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss=0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids=target_ids[:, :-1].contiguous()
            labels=target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:]==tokenizer.pad_token_id]=-100

            outputs=model(
                input_ids=source_ids,
                attention_mask=source_mask,
                decoder_input_ids=decoder_input_ids,
                labels=labels,
            )

            loss=outputs.loss
            val_loss+=loss

    val_loss=val_loss / len(dataloader)
    return val_loss


best_loss=10000
for epoch in range(epochs):
    train_loss=train(model, optimizer, train_dataloader)
    val_loss=evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        best_loss=val_loss
        torch.save(model.state_dict(), "../T5ForConditionalGeneration.pt")
        print("Saved the model weights")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1: Train Loss: 4.3188 Val Loss: 3.3270
Saved the model weights
Epoch 2: Train Loss: 3.4174 Val Loss: 2.9206
Saved the model weights
Epoch 3: Train Loss: 3.1428 Val Loss: 2.7673
Saved the model weights


In [8]:
#T5 생성 모델 테스트
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:
        generated_ids=model.generate(
            input_ids=source_ids,
            attention_mask=source_mask,
            max_length=128,
            num_beams=3,
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True,
        )

        for generated, target in zip(generated_ids, target_ids):
            pred=tokenizer.decode(
                generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            actual=tokenizer.decode(
                target, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            print("Generated Headline Text:", pred)
            print("Actual Headline Text   :", actual)
        break

Generated Headline Text: Clinton leads Trump by 4 percentage points in four-war race for Nov. 8 election.
Actual Headline Text   : Clinton leads Trump by 4 points in Washington Post: ABC News poll
Generated Headline Text: U.S. senators sharpen potential line of attack against Gorsuch's nomination to Supreme Court
Actual Headline Text   : Democrats question independence of Trump Supreme Court nominee
Generated Headline Text: u.S. warns Saudi Arabia over humanitarian situation in Yemen could constrain U.S. aid, official says.
Actual Headline Text   : In push for Yemen aid, U.S. warned Saudis of threats in Congress
Generated Headline Text: Romanian anti-corruption prosecutors open investigation into Liviu Dragnea accused of forming criminal group to siphon off cash from state projects.
Actual Headline Text   : Romanian ruling party leader investigated over 'criminal group'
Generated Headline Text: environmental activist endorses Hillary Clinton for U.S. president a day after she secured n